# HazeSignal: can fires and wind warn of haze earlier?

**Hypothesis:** more fire hotspots in Sumatra and Kalimantan, when winds carry air from those regions toward Kuala Lumpur, are associated with higher ground-level PM2.5 one or two days later.

This is a deliberately small September 2019 exploration. We will keep the intermediate tables visible and treat a messy result as useful evidence, not something to hide.

## 1. Load the small analysis helpers

The notebook uses the same TypeScript functions as the command-line analysis. Run it from the repository's `notebooks` folder with the Deno Jupyter kernel.

In [ ]:
import { parse } from "npm:csv-parse@7.0.2/sync";
import { combineDailyData, dailyWind, initialBearing, KUALA_LUMPUR, linearRegression, scatterSvg, SOURCE_CENTROIDS, windTravelBearing, alignmentScore } from "../src/analysis.ts";
const readCsv = async <T>(path: string): Promise<T[]> => parse(await Deno.readTextFile(path), { columns: true, skip_empty_lines: true, trim: true }) as T[];

## 2. Check the source files

FIRMS and OpenAQ require free personal keys, so secrets and authenticated responses are not committed. The Open-Meteo wind sample is included. Follow the README to create the two missing September 2019 files.

In [ ]:
const paths = {
  firms: "../data/firms_2019-09-01_2019-09-30.csv",
  wind: "../data/wind_2019-09-01_2019-09-30.csv",
  pm25: "../data/pm25_2019-09-01_2019-09-30.csv",
};

for (const [name, path] of Object.entries(paths)) {
  try { await Deno.stat(path); console.log(`✓ ${name}: ${path}`); }
  catch { console.log(`✗ ${name}: ${path} is missing`); }
}

## 3. Fire hotspots during the severe haze period

The FIRMS file contains one row per VIIRS detection. Counting rows by acquisition date gives our simplest fire-activity measure. This ignores fire intensity for now; later work should use the `frp` column as well.

In [ ]:
const hotspots = await readCsv<Record<string, string>>(paths.firms);
const fireCounts = Object.groupBy(hotspots, row => row.acq_date);
console.table(Object.entries(fireCounts).slice(0, 10).map(([date, rows]) => ({ date, hotspot_count: rows?.length ?? 0 })));

## 4. Wind direction and the source-to-Malaysia bearing

A compass bearing is calculated from each source centre to Kuala Lumpur. If `lat₁, lon₁` is the source and `lat₂, lon₂` is Kuala Lumpur, the initial bearing is:

`atan2(sin(Δlon) cos(lat₂), cos(lat₁) sin(lat₂) − sin(lat₁) cos(lat₂) cos(Δlon))`

Meteorological direction says where wind comes **from**, so smoke travel is `(wind direction + 180°) mod 360°`. The alignment score is `max(0, cos(travel bearing − route bearing))`. This makes the vector calculation visible instead of hiding it in a weather library.

In [ ]:
const routes = Object.entries(SOURCE_CENTROIDS).map(([region, source]) => ({
  region,
  bearing_to_kl: initialBearing(source.latitude, source.longitude, KUALA_LUMPUR.latitude, KUALA_LUMPUR.longitude),
}));
console.table(routes);

const workedExample = { wind_from: 225, smoke_travels_toward: windTravelBearing(225) };
console.log(workedExample, "Sumatra alignment:", alignmentScore(225, routes[0].bearing_to_kl));

In [ ]:
const rawWind = await readCsv<Record<string, string>>(paths.wind);
const wind = dailyWind(rawWind.map(row => ({
  date: row.date,
  wind_speed_kmh: Number(row.wind_speed_kmh),
  wind_direction_degrees: Number(row.wind_direction_degrees),
})));
console.table(wind.slice(0, 10));

## 5. Ground-level PM2.5

For the 2019 experiment this should be a DOE/APIMS export from the Cheras monitor with `date` and `pm25_ug_m3` columns. OpenAQ's present Kuala Lumpur record begins in 2022, so substituting it here would not validate the historical haze episode.

In [ ]:
const rawPm25 = await readCsv<Record<string, string>>(paths.pm25);
const pm25 = rawPm25.map(row => ({ date: row.date, pm25_ug_m3: Number(row.pm25_ug_m3) }));
console.table(pm25.slice(0, 10));

## 6. Combine each date with later PM2.5

For every wind date, we count hotspots, calculate how well the wind aligns for each hotspot's region, and look up PM2.5 on the next two calendar days. `aligned_hotspot_count` is the sum of the individual alignment scores; a fire contributes 1 when wind points directly toward Kuala Lumpur and 0 when it blows across or away.

In [ ]:
const combined = combineDailyData(hotspots.map(row => ({
  acq_date: row.acq_date,
  region: row.region as "sumatra" | "kalimantan",
})), wind, pm25);
console.table(combined.map(row => ({
  date: row.date, hotspot_count: row.hotspot_count, wind_alignment: row.wind_alignment.toFixed(2),
  aligned_hotspots: row.aligned_hotspot_count.toFixed(1), pm25: row.pm25_ug_m3,
  pm25_next_day: row.pm25_next_day, pm25_in_two_days: row.pm25_in_two_days,
})));

## 7. Simple next-day regression

We fit `next-day PM2.5 = intercept + slope × aligned hotspots`. Pearson's `r` describes the direction and strength of the straight-line association. `r²` is the fraction of variation described by this one-predictor line in this small sample; it is not forecast accuracy.

In [ ]:
const points = combined.flatMap(row => row.pm25_next_day == null ? [] : [{ x: row.aligned_hotspot_count, y: row.pm25_next_day }]);
const regression = linearRegression(points);
console.table(regression);
const svg = scatterSvg(points, regression);
await Deno.writeTextFile("../data/regression.svg", svg);
display({ "image/svg+xml": svg }, { raw: true });

## 8. Interpret the result honestly

A positive `r` would be consistent with the hypothesis, but one month cannot establish a useful warning signal. A weak or negative value is also informative: the route may be represented poorly, transport may take a different amount of time, rain may remove particles, or hotspot count may be too crude. Compare both one-day and two-day leads and report whichever is stronger only alongside the other result.

A fuller model needs several haze and non-haze seasons, fire radiative power, rainfall, humidity, boundary-layer height, wind along the transport route, fire persistence, and multiple Malaysian monitors. It also needs a held-out period to test whether apparent correlation predicts new days.

In [ ]:
const twoDayPoints = combined.flatMap(row => row.pm25_in_two_days == null ? [] : [{ x: row.aligned_hotspot_count, y: row.pm25_in_two_days }]);
const twoDay = linearRegression(twoDayPoints);
console.table([
  { lead: "1 day", r: regression.r, r_squared: regression.rSquared, observations: regression.observations },
  { lead: "2 days", r: twoDay.r, r_squared: twoDay.rSquared, observations: twoDay.observations },
]);